In [1]:
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

### Basic Qunatization Testing

In [2]:
fp32_values = np.sort(np.random.uniform(low = 0.0, high = 5.0, size = (10)))
print(fp32_values)

beta = np.max(fp32_values)
alpha = np.min(fp32_values)
beta_q = pow(2, 8) - 1
alpha_q = 0 

print(beta, alpha, beta_q, alpha_q)

[0.47908064 0.67628917 1.06521873 1.17896708 1.23958678 2.35833443
 2.91308574 3.20907596 3.51666015 4.76128434]
4.761284338671802 0.47908064162076824 255 0


In [3]:
# scale 
s = (beta - alpha) / (beta_q - alpha_q) 
print(s)

# zero point 
z = round ((beta * alpha_q - alpha * beta_q) / (beta - alpha))
print(z)

# clipping 
q = np.clip(z, alpha_q, beta_q)
print(q)

# calculating the q values 
q_values = np.clip((fp32_values / s) + z, alpha_q, beta_q).astype(np.uint8)
print(fp32_values)
print(q_values)

0.016792955674709932
-29
0
[0.47908064 0.67628917 1.06521873 1.17896708 1.23958678 2.35833443
 2.91308574 3.20907596 3.51666015 4.76128434]
[  0  11  34  41  44 111 144 162 180 254]


In [4]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fp32_text = [str(round(x,2)) for x in fp32_values.tolist()]
int8_text = [str(round(x,2)) for x in q_values.tolist()]    

fig = make_subplots(rows=1, cols=2)
fig.add_trace(go.Scatter(x=np.arange(len(fp32_values)), 
                         y=fp32_values, 
                         text=fp32_text,
                         mode='markers+text',
                         textposition='top left',
                         name='fp32_values'), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(len(q_values)),
                         y=q_values,
                         text=int8_text,
                         mode='markers+text',
                         textposition='bottom right', 
                         name='Quantized values'), row=1, col=2)    


In [8]:
# another way to quantize 

import numpy as np 

x = (np.random.rand(20) * 5 - 2.5)

frac_bits = 15 - np.ceil(np.log(np.max(np.abs(x))))
x_q = np.round(x * 2 ** frac_bits)
x_f = x_q / 2 ** frac_bits

print("x:")
print(x)
print("Quantized x:")
print(x_q)
print("Quantization error:")
print(x - x_f)  

x:
[-1.59065131 -2.16413377 -1.32063293 -1.32991345  0.81768595  1.23014521
  2.05434857  1.15856612 -0.35048357  0.18342235  2.40513299 -2.09867889
  2.24341562 -2.08896579  0.46084865 -0.42411171 -1.71209668  0.64075783
  1.77398359  1.04807663]
Quantized x:
[-26061. -35457. -21637. -21789.  13397.  20155.  33658.  18982.  -5742.
   3005.  39406. -34385.  36756. -34226.   7551.  -6949. -28051.  10498.
  29065.  17172.]
Quantization error:
[-1.41064079e-05 -1.02351795e-05 -1.52500894e-05 -1.84337924e-05
 -2.04269026e-06 -1.83654012e-05  2.72797169e-05 -3.21764036e-06
 -1.97062800e-05  1.17092585e-05 -1.83724850e-05  1.49542912e-05
  7.41595048e-06  2.34671140e-05 -2.78180096e-05  2.15919281e-05
  4.89671569e-07  1.07639079e-05 -3.23140222e-06 -1.90753441e-05]


### CMSIS-NN Quantization

In [9]:
"""
quantize numpy array 
return: 
q_x: quantized array 
"""

def quantize_array(x, bit_depth=8): 
    min_x = x.min()
    max_x = x.max()

    # find the number of integer bits to represent this range 
    int_bits = int(np.ceil(np.log2(max(abs(min_x), abs(max_x)))))
    if int_bits < 0: int_bits = 0 
    # remaining bits are fractional bits 
    frac_bits = bit_depth - 1 - int_bits 
    # floating point weights are scaled and rounded to [-2^(bit-1), 2^(bit-1)-1], 
    # which are used in the fixed point operations on the actual hardware 
    q_x = np.round(x* (2 ** frac_bits))
    # to quantify the impact of quanitzed weights, scale them back to 
    # original range to run inference using quantized weights 
    fp_x =  q_x / (2 ** frac_bits)
    
    return q_x, fp_x, int_bits, frac_bits


"""
compute frac_bits based on max and min list 
used for activation(feature map) quantization 
"""
def min_max_quantize(max_list, min_list, bit_depth=8):
    frac_list = [] 
    for i in range(len(max_list)):
        int_bits = int(np.ceil(np.log2(max(abs(min_list[i])))))
        if int_bits < 0: int_bits = 0
        # remaining bits are fractional bits (1-bit for sign)
        frac_bits = bit_depth - 1 - int_bits
        frac_list.append(frac_bits) 
    return frac_list


In [10]:
import numpy as np 
import tensorflow as tf 
from tensorflow.keras.layers import Conv2D, MaxPool2D, Input, ReLU, GlobalAveragePooling2D, Flatten, Dense, Activation  
from tensorflow.keras import Model 

### Original Floating-point Model 

In [12]:
def Net(): 
    x_in = Input(shape=(4096, 1, 1))

    x = Conv2D(filters=16, kernel_size=(9, 1), strides=1, padding='valid')(x_in)
    x = ReLU()(x)
    x = Conv2D(filters=16, kernel_size=(9, 1), strides=1, padding='valid')(x)
    x = ReLU()(x)
    x = MaxPool2D(pool_size=(16, 1))(x)

    x = Conv2D(filters=32, kernel_size=(3, 1), strides=1, padding='valid')(x)
    x = ReLU()(x)
    x = Conv2D(filters=32, kernel_size=(3, 1), strides=1, padding='valid')(x)
    x = ReLU()(x)
    x = MaxPool2D(pool_size=(4, 1))(x)
    
    x = Conv2D(filters=32, kernel_size=(3, 1), strides=1, padding='valid')(x)
    x = ReLU()(x)
    x = Conv2D(filters=32, kernel_size=(3, 1), strides=1, padding='valid')(x)
    x = ReLU()(x)
    x = MaxPool2D(pool_size=(4, 1 ))(x)
    
    x = Conv2D(filters=64, kernel_size=(3, 1), strides=1, padding='valid')(x)
    x = ReLU()(x)
    x = Conv2D(filters=64, kernel_size=(3, 1), strides=1, padding='valid')(x)
    x = ReLU()(x)
    x = MaxPool2D(pool_size=(4, 1))(x)
    
    x = GlobalAveragePooling2D()(x)
    x = Flatten()(x)
    
    x = Dense(64)(x)
    x = ReLU()(x)
    x = Dense(512)(x)
    x = ReLU()(x)
    x = Dense(1)(x)
    x_out = Activation('sigmoid')(x)
    return Model(x_in, x_out)

model = Net()

# Load the dataset
x = np.load("./data/input.npy")
y = np.load("./data/target.npy")
x = np.expand_dims(np.expand_dims(np.squeeze(x), -1), -1)

print("Input shape: ", x.shape)
print("Output shape: ", y.shape)

# Load pre-trained weight 
model.load_weights("./weights/tf.h5")

# Evaluation (float32)
length = y.shape[0]
positives = 0
for i in range(length):
    pred = model(x[i:i+1])
    pred = (pred > 0.5)
    positives += np.sum(y[i]==pred)

print("Accuracy: ", positives / length)

FileNotFoundError: [Errno 2] No such file or directory: './data/input.npy'